In [ ]:
# ============================================================================
# INTERAKTÍV PREDIKCIÓ - ipywidgets
# ============================================================================

import ipywidgets as widgets
from IPython.display import display, HTML
import pickle
import json

# ============================================================================
# Modell és feature lista betöltése
# ============================================================================

# FONTOS: Cseréld ki a fájlneveket a sajátodra!
MODEL_FILE = 'models/logistic_regression_csgo_20251019_101234.pkl'
FEATURES_FILE = 'models/feature_list_20251019_101234.json'

with open(MODEL_FILE, 'rb') as f:
    loaded_model = pickle.load(f)

with open(FEATURES_FILE, 'r') as f:
    feature_data = json.load(f)
    required_features = feature_data['features']

print(f"✅ Modell betöltve: {MODEL_FILE}")
print(f"✅ Feature-ök ({len(required_features)}):\n   {', '.join(required_features[:5])}...")

# ============================================================================
# Widget-ek létrehozása minden feature-höz
# ============================================================================

feature_widgets = {}

# Tipikus value range-ek
default_values = {
    'winrate': 0.5,
    'rank': 10,
    'rating': 1.0,
    'adr': 75.0,
    'swing': 0.0,
    'odds': 2.0,
    'implied': 0.5,
    'diff': 0.0,
    'h2h': 0.5,
    'games': 5,
    'score': 1.0,
    'streak': 0,
    'pickrate': 0.5,
    'point': 0,
    'std': 0.2
}

# Feature-önként widget létrehozása
for feat in required_features:
    # Alapértelmezett érték felismerése
    default = 0.0
    for key, val in default_values.items():
        if key in feat.lower():
            default = val
            break
    
    # FloatText widget
    feature_widgets[feat] = widgets.FloatText(
        value=default,
        description=feat[:25] + '...' if len(feat) > 25 else feat,
        disabled=False,
        style={'description_width': '250px'},
        layout=widgets.Layout(width='400px')
    )

# ============================================================================
# Odds input (külön, mivel ez NEM feature ha odds nélküli modell)
# ============================================================================

home_odds_input = widgets.FloatText(
    value=1.75,
    description='Home Odds:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='300px')
)

away_odds_input = widgets.FloatText(
    value=2.10,
    description='Away Odds:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='300px')
)

edge_threshold_input = widgets.FloatSlider(
    value=0.02,
    min=0.0,
    max=0.20,
    step=0.01,
    description='Edge Threshold:',
    style={'description_width': '150px'},
    layout=widgets.Layout(width='400px'),
    readout_format='.2f'
)

# Predict gomb
predict_button = widgets.Button(
    description='🎯 PREDICT & VALUE BET',
    button_style='success',
    layout=widgets.Layout(width='400px', height='50px')
)

# Output area
output = widgets.Output()

# ============================================================================
# Predikciós logika
# ============================================================================

def on_predict_click(b):
    with output:
        output.clear_output()
        
        # Feature-ök begyűjtése
        feature_values = {feat: widget.value for feat, widget in feature_widgets.items()}
        
        # DataFrame létrehozása
        import pandas as pd
        new_match_df = pd.DataFrame([feature_values])
        new_match_df = new_match_df[required_features]  # Sorrend fontos!
        
        # Predikció
        pred_home_win = loaded_model.predict_proba(new_match_df)[0, 1]
        pred_away_win = 1 - pred_home_win
        
        # Odds és edge kalkuláció
        home_odds = home_odds_input.value
        away_odds = away_odds_input.value
        home_implied = 1 / home_odds
        away_implied = 1 / away_odds
        
        edge_home = pred_home_win - home_implied
        edge_away = pred_away_win - away_implied
        edge_th = edge_threshold_input.value
        
        # Betting decision
        bet_decision = "❌ NO VALUE BET"
        bet_color = "red"
        bet_details = ""
        
        if edge_home > edge_th:
            bet_decision = "✅ BET ON HOME!"
            bet_color = "green"
            expected_value = (pred_home_win * home_odds - 1) * 100
            bet_details = f"""
            <b>Odds:</b> {home_odds}<br>
            <b>Edge:</b> {edge_home*100:.2f}%<br>
            <b>Expected Value:</b> {expected_value:.2f}%
            """
        elif edge_away > edge_th:
            bet_decision = "✅ BET ON AWAY!"
            bet_color = "green"
            expected_value = (pred_away_win * away_odds - 1) * 100
            bet_details = f"""
            <b>Odds:</b> {away_odds}<br>
            <b>Edge:</b> {edge_away*100:.2f}%<br>
            <b>Expected Value:</b> {expected_value:.2f}%
            """
        else:
            bet_details = f"""
            <b>Home Edge:</b> {edge_home*100:.2f}% (threshold: {edge_th*100:.0f}%)<br>
            <b>Away Edge:</b> {edge_away*100:.2f}% (threshold: {edge_th*100:.0f}%)
            """
        
        # HTML output
        html_output = f"""
        <div style="border: 3px solid {bet_color}; padding: 20px; border-radius: 10px; background-color: #f9f9f9;">
            <h2 style="color: {bet_color}; text-align: center;">{bet_decision}</h2>
            <hr>
            <h3>📊 Prediction:</h3>
            <table style="width: 100%; font-size: 16px;">
                <tr>
                    <td><b>Home Win Probability:</b></td>
                    <td style="text-align: right;"><span style="font-size: 20px; color: {'green' if pred_home_win > 0.5 else 'black'};">{pred_home_win:.2%}</span></td>
                </tr>
                <tr>
                    <td><b>Away Win Probability:</b></td>
                    <td style="text-align: right;"><span style="font-size: 20px; color: {'green' if pred_away_win > 0.5 else 'black'};">{pred_away_win:.2%}</span></td>
                </tr>
            </table>
            <hr>
            <h3>💰 Odds & Implied Probability:</h3>
            <table style="width: 100%; font-size: 16px;">
                <tr>
                    <td><b>Home Odds:</b></td>
                    <td style="text-align: right;">{home_odds} <span style="color: gray;">({home_implied:.2%})</span></td>
                </tr>
                <tr>
                    <td><b>Away Odds:</b></td>
                    <td style="text-align: right;">{away_odds} <span style="color: gray;">({away_implied:.2%})</span></td>
                </tr>
            </table>
            <hr>
            <h3>🎯 Betting Decision:</h3>
            {bet_details}
        </div>
        """
        
        display(HTML(html_output))

predict_button.on_click(on_predict_click)

# ============================================================================
# UI Layout
# ============================================================================

# Feature input-ok 2 oszlopban
left_features = list(feature_widgets.values())[:len(feature_widgets)//2]
right_features = list(feature_widgets.values())[len(feature_widgets)//2:]

left_column = widgets.VBox(left_features)
right_column = widgets.VBox(right_features)

feature_inputs = widgets.HBox([left_column, right_column])

# Odds és threshold
odds_section = widgets.VBox([
    widgets.HTML("<h3>📈 Odds (nem feature, csak value bet kalkulációhoz):</h3>"),
    home_odds_input,
    away_odds_input,
    edge_threshold_input
])

# Teljes UI
full_ui = widgets.VBox([
    widgets.HTML("<h1 style='text-align: center; color: #2c3e50;'>🎮 CS:GO Value Bet Predictor</h1>"),
    widgets.HTML("<hr>"),
    widgets.HTML("<h2>⚙️ Feature Input:</h2>"),
    feature_inputs,
    widgets.HTML("<hr>"),
    odds_section,
    widgets.HTML("<br>"),
    predict_button,
    widgets.HTML("<br>"),
    output
])

display(full_ui)